# Figure S7: Bayesian position decoding (C/P)

Panels **S7b and S7c**: posterior matrices and decoding error.

This notebook follows **Xu et al., manuscript dated 30 October 2026**. Data loading, model fitting, panel selection and plotting are shown below as executable steps. Shared scientific methods remain in `src/`; this notebook does not delegate the figure to `run_figure()`.

Start with the separate manuscript source-data files listed in `data/required_files.csv`. These are analysis source tables and frozen map/embedding arrays, rather than raw movies. The aligned-data computation examples are in `00_recompute_metrics.ipynb`.

Run cells from top to bottom. `REFIT_STATISTICS=True` refits the manuscript models; set it to `False` to use the frozen estimates. Outputs are saved separately from inputs.

## 1. Paths and execution parameters

Set `PPRH_DATA_DIR` or edit `DATA_ROOT`. The default refits statistical models. No raw neural resampling is triggered here.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "main.py").is_file() and (p / "src/paper.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, display
from paper_io import read_table, save_panel as _save_panel, write_tables, p_symbol, write_run_record

DATA_ROOT = Path(os.environ.get("PPRH_DATA_DIR", str(ROOT / "data/input")))
OUTPUT_ROOT = Path(os.environ.get("PPRH_OUTPUT_DIR", str(ROOT / "outputs/notebooks")))
REFIT_STATISTICS = os.environ.get("PPRH_REFIT_STATISTICS", "1") == "1"
FORMATS = ("svg", "png")
DPI = 300

data_root = DATA_ROOT
output = OUTPUT_ROOT / "FigS7"
refit, formats, dpi = REFIT_STATISTICS, FORMATS, DPI
output.mkdir(parents=True, exist_ok=True)
if not data_root.is_dir():
    raise FileNotFoundError("Supply the separate manuscript inputs through DATA_ROOT; see data/README.md.")


## 2. Analysis contract

The Gaussian Bayesian decoder uses 98 physical spatial bins, five complete-trial folds, 50 fixed neurons per mouse across days and 100 within-trial circular label shifts. These cells start from its completed output tables and soft posteriors. Posterior matrices average within true bin per mouse before averaging mice equally; shuffle errors are averaged within mouse/day rather than treated as biological replicates. Error plots use cm; the original shared inference helper's native effect/SE/CI columns use mm. The statistical model retains Day and uses no movement covariates. Numerical decoder training is shown separately in notebook 00.

## 3. Notebook functions

The export function saves and previews each panel. Where needed, the source-table loader or annotation helper is defined here in full. Numerical estimators and shared plotting functions are imported in the relevant analysis cells.

In [ ]:
def save_panel(fig, output_root, panel, description, **kwargs):
    """Save one panel with the shared export method, then preview its result."""
    paths = _save_panel(fig, output_root, panel, description, **kwargs)
    svg = next((p for p in paths if p.endswith(".svg")), None)
    png = next((p for p in paths if p.endswith(".png")), None)
    if svg:
        display(SVG(filename=svg))
    elif png:
        display(Image(filename=png))
    return paths


## 4. Load the decoder outputs

In [ ]:
from figure_helpers import bayesian_plots as h
from maze_analysis.stats import build_group_day_summary
from maze_analysis.plotting import add_project_colorbar
from maze_analysis.position_decoding_summary import fit_position_inference

comparison, panel = "CP", "FigS7"
exports = []
with np.load(DATA_ROOT / f"decode_{comparison}/matrices.npz") as z:
    groups = z["groups"].astype(str)
    days = z["days"]
    matrices = z["mean_posterior"]
display(pd.DataFrame({"group": groups}))
print("Recording days:", days, "Posterior matrix shape:", matrices.shape)


## 5. S7b — mean posterior by group and day

The shared color ceiling is computed once for this comparison. Rows and columns retain the true/candidate spatial-bin interpretation used by the original plotting helper.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(13.5, 5.5), layout="constrained")
vmax = h.ceiling(matrices, .45)
for i, group in enumerate(groups):
    for j, day in enumerate(days):
        image = h.heatmap(axes[i, j], matrices[i, j], f"{group} D{day}", vmax, "Candidate position bin")
add_project_colorbar(fig, image, axes, label="Mean posterior probability (linear)", ticks=np.linspace(0, vmax, 6))
exports += save_panel(fig, output, f"{panel}_b", "mean_posterior", formats=formats, dpi=dpi)


## 6. Retain mouse-day errors and summarize mice

Only the Bayesian and circular-shift models are selected. No KNN results are plotted, and the day dimension remains explicit.

In [ ]:
daily = read_table(DATA_ROOT, f"decode_{comparison}/mouse_day.csv")
daily = daily[daily.model.isin(("Bayes", "Shift control"))].copy()
long = daily.melt(id_vars=["group", "mouse", "day", "model"], value_vars=["euclidean_mean_cm"],
                  var_name="metric", value_name="value")
summary = build_group_day_summary(long, "value", by=["model", "metric"])
display(long.head())
display(summary.head())


## 7. S7c — decoding error across days

In [ ]:
h.PLOT_GROUPS = tuple(groups)
fig, ax = plt.subplots(figsize=(4.2, 3.5), layout="constrained")
h.performance(ax, "euclidean_mean_cm", long, summary, days)
ax.set_ylabel("Decoding error (cm)")
exports += save_panel(fig, output, f"{panel}_c", "Euclidean_decoding_error", formats=formats, dpi=dpi)


## 8. Refit the original longitudinal inference

`REFIT_STATISTICS` controls refitting from the released model-input table. The native effect-size unit convention is retained; the output is not rescaled silently. With refitting disabled, the corresponding reference estimates remain in the input dataset.

In [ ]:
pattern = f"model_input*{'C-I' if comparison == 'CI' else 'C-P'}.csv"
model_paths = list((DATA_ROOT / "decode_statistics").glob(pattern))
if len(model_paths) != 1:
    raise ValueError("Expected one matching decoding model-input table")
inputs = pd.read_csv(model_paths[0])
inputs["metric"] = "distance_cm"
display(inputs.head())
if REFIT_STATISTICS:
    details = fit_position_inference(inputs, stat_groups=tuple(groups), stat_metrics=("distance_cm",), return_details=True)
    write_tables(details, output / "statistics" / panel)
    display(details["inference"].head())
else:
    print("Refitting disabled; use the supplied reference-estimate tables for inference.")


## 9. Export record

In [ ]:
write_run_record(output / "run.json", figure="S7", refit=refit, exports=exports)
display(pd.DataFrame({"exported_file": [str(Path(p).relative_to(output)) for p in exports]}))
